# Control A — coarse-output intact tasks

`DECISION_control_A.md` §4.5 specified the intact side as pretraining-corpus top-1 match **plus** public tasks the paper found unaffected — MMLU, SQuAD, sentiment, CoLA, "any subset is adequate; two is enough".

Only the corpus half was built for the first run. It **failed**: WikiText top-1 = 0.577 at heavy, against a pre-registered threshold of 0.90.

**That failure stands and nothing here changes it.** But it was measured on the most *sensitive* of the specified measures. Next-token match over 151,936 tokens registers any distributional shift; a 4-way multiple choice only changes if the shift reorders four specific tokens. That asymmetry is almost certainly why the paper reported those tasks as intact — and it means the two measures answer different questions.

Running these completes an under-delivered spec. They were named in the signed document from the start, so this is not shopping for a measure that passes. Results are **supplementary**.

### The gate built into this

MMLU is 25% at chance, SST-2 is 50%. A model near chance cannot be degraded, so "still at chance after ablation" is not evidence of preservation. Each task is checked for headroom first and **excluded from the verdict if it fails** — the same logic as `diagnose()`'s NO EFFECT branch.

**Cost:** roughly 25–35 minutes, ~8 units. Runtime → **A100**.

## Cell 1 — Setup

In [ ]:
import os, sys, subprocess, torch

!pip -q install -U transformers accelerate huggingface_hub datasets

if not os.path.isdir("/content/jacobian-lens"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/anthropics/jacobian-lens.git"],
                   cwd="/content", check=True)

os.makedirs("/content/ablation", exist_ok=True)
open("/content/ablation/__init__.py","a").close()
for p in ("/content/jacobian-lens", "/content"):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = "/content/jacobian-lens:/content"

import importlib; importlib.invalidate_caches()
import jlens
print("jlens OK |", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## Cell 2 — Write `ablation/directions.py`

In [ ]:
%%writefile ablation/directions.py
"""Direction selection and subspace projection for J-space / R-space ablation.

Implements the direction-set half of the ablation harness. Every selector here
produces a set of residual-stream directions of a specified size; the harness
projects them out. Keeping selection separate from projection is what makes the
matched controls of proposal 4.8 cheap: same projection, different selector.

Terminology (guide 2.1): nothing here is "the workspace". These are candidate
directions until Phase 3 says otherwise.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch


# --- J-lens vector construction -------------------------------------------

def lens_vectors(
    unembed_weight: torch.Tensor, jacobian: torch.Tensor, token_ids: torch.Tensor
) -> torch.Tensor:
    """The J-lens vectors for ``token_ids`` at one layer.

    Paper §2.1 defines the J-lens vectors as the rows of ``W_U J_l``. Only the
    requested rows are materialised: the full product is ``[vocab, d_model]``
    and is far too large to hold for a real vocabulary.

    Args:
        unembed_weight: ``W_U``, shape ``[vocab, d_model]``.
        jacobian: ``J_l``, shape ``[d_model, d_model]``.
        token_ids: Shape ``[..., k]``.

    Returns:
        Shape ``[..., k, d_model]``.
    """
    rows = unembed_weight.index_select(0, token_ids.reshape(-1).to(unembed_weight.device))
    rows = rows.to(jacobian.dtype) @ jacobian
    return rows.reshape(*token_ids.shape, jacobian.shape[-1])


# --- Selectors -------------------------------------------------------------

def select_by_rank(
    lens_logits: torch.Tensor,
    k: int,
    *,
    rank_offset: int = 0,
    excluded: torch.Tensor | None = None,
) -> torch.Tensor:
    """Token ids ranked ``rank_offset .. rank_offset + k`` by lens score.

    ``rank_offset=0`` gives the top-k the paper ablates. ``rank_offset=k`` gives
    the next-k, which is the "matched but not selected" control: same lens, same
    size, adjacent rank band. A candidate subspace that matters no more than the
    next-k has not earned H1 (proposal 4.8, extended per the probe-swap design).

    Args:
        lens_logits: Shape ``[n_positions, vocab]``.
        k: Number of directions.
        rank_offset: Rank to start from.
        excluded: Boolean mask ``[n_positions, vocab]``; True entries are never
            selected. This carries the clean-pass exclusion — see
            :func:`clean_top_mask`.

    Returns:
        Shape ``[n_positions, k]``.
    """
    scores = lens_logits.clone()
    if excluded is not None:
        scores = scores.masked_fill(excluded, float("-inf"))
    top = scores.topk(rank_offset + k, dim=-1).indices
    return top[:, rank_offset:]


def random_lens_tokens(
    n_positions: int,
    k: int,
    vocab_size: int,
    generator: torch.Generator,
    *,
    excluded: torch.Tensor | None = None,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Uniformly random token ids — matched-size random control (proposal 4.8).

    Drawn from the lens dictionary rather than isotropically, so the control
    asks "is it *these* lens directions, or any lens directions?". Strictly the
    harder question of the two; run both.
    """
    out = torch.empty(n_positions, k, dtype=torch.long, device=device)
    for p in range(n_positions):
        while True:
            cand = torch.randint(
                vocab_size, (k,), generator=generator, device=generator.device
            ).to(device)
            if excluded is None or not bool(excluded[p, cand].any()):
                out[p] = cand
                break
    return out


def random_isotropic(
    n_positions: int, k: int, d_model: int, generator: torch.Generator,
    *, device: torch.device | None = None, dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    """Isotropic random unit directions — the paper's random-direction control."""
    v = torch.randn(
        n_positions, k, d_model, generator=generator, device=generator.device,
        dtype=torch.float32,
    ).to(device=device, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def clean_top_mask(
    clean_next_token_logits: torch.Tensor, n_exclude: int = 10
) -> torch.Tensor:
    """Mask marking the clean pass's top-``n_exclude`` predictions per position.

    **This is the confound guard, and it is not optional.** Paper: "we do not
    ablate any tokens that appear in the top-10 tokens of a clean forward pass,
    so as to specifically target the J-space's effects on internal reasoning
    rather than report." Without it, ablation suppresses whatever the model was
    about to say, performance drops for a trivial reason, and H1 gets
    "confirmed" by an artifact.

    Args:
        clean_next_token_logits: Shape ``[n_positions, vocab]`` from an
            unablated forward pass.

    Returns:
        Boolean ``[n_positions, vocab]``, True where a token must not be ablated.
    """
    mask = torch.zeros_like(clean_next_token_logits, dtype=torch.bool)
    top = clean_next_token_logits.topk(n_exclude, dim=-1).indices
    return mask.scatter(-1, top, True)


# --- Projection ------------------------------------------------------------

@dataclass(frozen=True)
class Basis:
    """An orthonormal-row basis with a validity mask for rank-deficient sets."""

    rows: torch.Tensor   # [n_positions, k, d_model], orthonormal rows
    keep: torch.Tensor   # [n_positions, k], float 1/0
    rank: torch.Tensor   # [n_positions], effective rank actually removed


def orthonormalise(vectors: torch.Tensor, *, rtol: float = 1e-6) -> Basis:
    """Orthonormal basis for the span of each position's direction set.

    J-lens vectors are overcomplete and non-orthogonal (paper §2.3), so a set of
    k of them may span fewer than k dimensions. Small singular values are masked
    out rather than dropped, which keeps the operation batched and makes the
    effective rank observable — report it, because "we ablated k directions" is
    false if the span was smaller.
    """
    vectors = vectors.to(torch.float32)
    _, s, vh = torch.linalg.svd(vectors, full_matrices=False)
    keep = (s > rtol * s[..., :1].clamp_min(1e-30)).to(vectors.dtype)
    return Basis(rows=vh, keep=keep, rank=keep.sum(-1))


def project_out(
    hidden: torch.Tensor, basis: Basis, *, mode: str = "subspace"
) -> torch.Tensor:
    """Remove the component of ``hidden`` inside the spanned subspace.

    Args:
        hidden: Shape ``[n_positions, d_model]``.
        mode: ``"subspace"`` projects onto the orthogonal complement of the span
            in one step. ``"sequential"`` removes each direction in turn, which
            is order-dependent for non-orthogonal vectors and therefore removes
            *less* than the full span.

    The paper's phrasing — "zero out the residual stream's projection onto
    each" — does not disambiguate these, and for non-orthogonal J-lens vectors
    they differ. ``"subspace"`` is the default because it is the one that
    actually removes the content; ``"sequential"`` is provided so the choice can
    be tested rather than assumed. Record which was used.
    """
    h = hidden.to(torch.float32)
    if mode == "subspace":
        coeffs = torch.einsum("prd,pd->pr", basis.rows, h) * basis.keep
        return (h - torch.einsum("pr,prd->pd", coeffs, basis.rows)).to(hidden.dtype)
    if mode == "sequential":
        for i in range(basis.rows.shape[1]):
            v = basis.rows[:, i, :] * basis.keep[:, i : i + 1]
            h = h - (h * v).sum(-1, keepdim=True) * v
        return h.to(hidden.dtype)
    raise ValueError(f"unknown mode {mode!r}")

## Cell 3 — Write `ablation/harness.py`

In [ ]:
%%writefile ablation/harness.py
"""Two-pass ablation harness.

Pass 1 is a clean forward pass: it records the residual stream at every band
layer, computes lens logits, and captures the clean next-token distribution.
Pass 2 re-runs with the selected directions projected out.

Two passes are not an optimisation choice — the confound guard of proposal 4.4
(paper: exclude the clean pass's top-10) *requires* knowing the clean output
before choosing what to ablate.

This harness is built to Phase 3 requirements from the first line, per guide
§3a: Control A, Control B, and the Phase 3 sweep all run through it unchanged.
"""

from __future__ import annotations

from dataclasses import dataclass, field, asdict
from typing import Any, Literal, Sequence

import torch

from jlens.hooks import ActivationRecorder
from jlens.lens import JacobianLens

from .directions import (
    Basis,
    clean_top_mask,
    lens_vectors,
    orthonormalise,
    project_out,
    random_isotropic,
    random_lens_tokens,
    select_by_rank,
)

Selector = Literal["topk", "next_k", "random_lens", "random_iso", "none"]


def record_at_or(spec, final):
    return sorted({*spec.layers, final})


def _mask_from_ids(ids: torch.Tensor, shape) -> torch.Tensor:
    """Rebuild the clean-top-k boolean mask from cached ids."""
    m = torch.zeros(shape, dtype=torch.bool, device=ids.device)
    return m.scatter(-1, ids, True)


@dataclass(frozen=True)
class AblationSpec:
    """One fully-specified ablation condition. Serialise this into every result.

    Attributes:
        layers: Band of block indices to ablate at. Light/medium/heavy differ
            only here — the paper varies the layer range, not k.
        k: Directions removed per position (proposal 4.7 sweeps this; the paper
            fixed it at 10). Sweeping k *and* layers multiplies runs — state
            which axis in prereg_phase3.md.
        selector: Which directions. ``"none"`` is the clean baseline.
        seed: Required for every random selector (guide §1.2).
        exclude_clean_top: Confound guard size. **Do not set to 0** except as a
            deliberate, logged demonstration of the artifact it prevents.
        mode: Projection mode; see :func:`project_out`.
        positions: Token positions to ablate at; ``None`` means all.
    """

    layers: tuple[int, ...]
    k: int
    selector: Selector = "topk"
    seed: int | None = None
    exclude_clean_top: int = 10
    mode: str = "subspace"
    positions: tuple[int, ...] | None = None

    def __post_init__(self) -> None:
        if self.selector in ("random_lens", "random_iso") and self.seed is None:
            raise ValueError(
                "random selectors require an explicit seed — an unseeded "
                "matched-random baseline is not reproducible and proposal 4.8 "
                "results computed against it are not reportable"
            )

    def key(self) -> str:
        import hashlib, json
        blob = json.dumps(asdict(self), sort_keys=True, default=str)
        return hashlib.sha256(blob.encode()).hexdigest()[:16]


@dataclass
class AblationResult:
    logits: torch.Tensor              # [n_positions, vocab] ablated next-token logits
    clean_logits: torch.Tensor        # [n_positions, vocab] unablated
    effective_rank: dict[int, torch.Tensor] = field(default_factory=dict)
    spec: AblationSpec | None = None
    ids: torch.Tensor | None = None       # [1, seq_len]; needed to score against
                                          # true next tokens on a corpus (intact side)


def prepare_lens(lens: JacobianLens, device) -> JacobianLens:
    """Move the Jacobians onto the compute device. **Does not touch dtype.**

    ``JacobianLens.__init__`` does ``J.float()`` on every Jacobian, so the class
    holds float32 regardless of what was on disk (``save`` writes fp16 purely
    for compactness). ``apply()`` correspondingly casts residuals with
    ``.float()`` before ``transport``. The library's internal contract is
    float32 throughout, and ``HFLensModel.unembed`` casts to the head's dtype
    itself, so nothing downstream needs the model's dtype here.

    An earlier version of this function cast the Jacobians to the model's dtype.
    That broke the contract and produced
    ``RuntimeError: expected mat1 and mat2 to have the same dtype`` inside
    ``transport``. Callers passing activations straight from
    ``ActivationRecorder`` (which are in the *model's* dtype, not float32) must
    cast those to float — see :func:`build_cache`.

    Only the device move is needed: without it ``transport`` copies a
    ``[d_model, d_model]`` matrix host-to-device on every call.
    """
    lens.jacobians = {k: v.to(device=device) for k, v in lens.jacobians.items()}
    return lens


@dataclass
class PromptCache:
    """Per-prompt work that every condition would otherwise repeat.

    The clean forward pass and the lens readout are identical across all
    conditions for a given prompt — only the direction *selection* differs. A
    37-condition sweep without this recomputes both 37 times.

    What is cached is deliberately small: the ranked token ids per layer, not
    the lens logits themselves. Lens logits are ``[n_positions, vocab]``, which
    at a 150k vocabulary is megabytes per layer per prompt; the ranked ids are
    ``[n_positions, k_max]``. Any ``k <= k_max`` is then a slice.

    ``k_max`` must be at least ``2 * max(k)`` in the sweep, because the
    ``next_k`` selector reads ranks ``k..2k``.
    """

    ids: torch.Tensor
    n_pos: int
    clean_logits: torch.Tensor
    excluded_ids: torch.Tensor | None          # [n_pos, n_exclude]
    ranked_ids: dict[int, torch.Tensor]        # layer -> [n_pos, k_max]
    k_max: int


@torch.no_grad()
def build_cache(
    model: Any, lens: JacobianLens, prompt: str, layers: Sequence[int],
    *, k_max: int, exclude_clean_top: int = 10, max_seq_len: int = 512,
) -> PromptCache:
    """Run the clean pass once and rank directions once, for reuse."""
    ids = model.encode(prompt, max_length=max_seq_len)
    final = model.n_layers - 1
    record_at = sorted({*layers, final})

    with ActivationRecorder(model.layers, record_at) as rec:
        model.forward(ids)
        acts = {i: rec.activations[i][0].detach() for i in record_at}
    clean_logits = model.unembed(acts[final])

    excluded = (clean_top_mask(clean_logits, exclude_clean_top)
                if exclude_clean_top > 0 else None)
    excluded_ids = (clean_logits.topk(exclude_clean_top, dim=-1).indices
                    if exclude_clean_top > 0 else None)

    ranked = {}
    for layer in layers:
        # .float() to match the lens's float32 Jacobians, as apply() does.
        lens_logits = model.unembed(lens.transport(acts[layer].float(), layer))
        ranked[layer] = select_by_rank(lens_logits, k_max, excluded=excluded)

    return PromptCache(ids, ids.shape[1], clean_logits, excluded_ids, ranked, k_max)


class _Ablator:
    """Forward hooks that project out precomputed per-position direction sets."""

    def __init__(
        self,
        blocks: Sequence[torch.nn.Module],
        bases: dict[int, Basis],
        position_mask: torch.Tensor | None,
        mode: str,
    ) -> None:
        self._blocks, self._bases = blocks, bases
        self._position_mask, self._mode = position_mask, mode
        self._handles: list[Any] = []

    def _hook(self, index: int):
        basis = self._bases[index]

        def fn(module, inputs, output):
            is_tuple = not torch.is_tensor(output)
            tensor = output[0] if is_tuple else output
            # tensor: [batch, seq, d_model]; harness runs batch=1.
            h = tensor[0]
            new = project_out(h, basis, mode=self._mode)
            if self._position_mask is not None:
                new = torch.where(self._position_mask[:, None], new, h)
            tensor = torch.cat([new[None], tensor[1:]], dim=0)
            return (tensor, *output[1:]) if is_tuple else tensor

        return fn

    def __enter__(self):
        try:
            for i in self._bases:
                self._handles.append(self._blocks[i].register_forward_hook(self._hook(i)))
        except Exception:
            self.__exit__()
            raise
        return self

    def __exit__(self, *exc) -> None:
        for h in self._handles:
            h.remove()
        self._handles = []


@torch.no_grad()
def run_ablation(
    model: Any,
    lens: JacobianLens,
    unembed_weight: torch.Tensor,
    prompt: str,
    spec: AblationSpec,
    *,
    max_seq_len: int = 512,
    cache: PromptCache | None = None,
) -> AblationResult:
    """Run one ablation condition end to end.

    Args:
        model: Anything satisfying ``jlens.protocol.LensModel``.
        lens: Fitted lens. ``spec.layers`` must be a subset of its source layers
            for lens-based selectors.
        unembed_weight: ``W_U``, ``[vocab, d_model]`` — usually
            ``model.lm_head.weight``. Passed explicitly because the LensModel
            protocol exposes ``unembed()`` (norm + head) but not ``W_U`` itself.
    """
    final = model.n_layers - 1

    # --- Pass 1: clean (skipped entirely when a cache is supplied) ---
    if cache is not None:
        if spec.k * (2 if spec.selector == "next_k" else 1) > cache.k_max:
            raise ValueError(
                f"cache holds k_max={cache.k_max} ranked directions but this "
                f"condition needs {spec.k * (2 if spec.selector == 'next_k' else 1)}. "
                "Rebuild the cache with a larger k_max."
            )
        ids, n_pos = cache.ids, cache.n_pos
        clean_logits, acts = cache.clean_logits, None
    else:
        ids = model.encode(prompt, max_length=max_seq_len)
        n_pos = ids.shape[1]
        with ActivationRecorder(model.layers, sorted({*spec.layers, final})) as rec:
            model.forward(ids)
            acts = {i: rec.activations[i][0].detach() for i in record_at_or(spec, final)}
        clean_logits = model.unembed(acts[final])

    if spec.selector == "none":
        return AblationResult(clean_logits, clean_logits, spec=spec, ids=ids)

    excluded = None
    if spec.exclude_clean_top > 0:
        excluded = (
            _mask_from_ids(cache.excluded_ids, clean_logits.shape)
            if cache is not None
            else clean_top_mask(clean_logits, spec.exclude_clean_top)
        )

    # --- Direction selection, per band layer ---
    gen = torch.Generator(device="cpu")
    if spec.seed is not None:
        gen.manual_seed(spec.seed)
    bases: dict[int, Basis] = {}
    ref = clean_logits
    for layer in spec.layers:
        h = acts[layer] if acts is not None else ref
        if spec.selector == "random_iso":
            vecs = random_isotropic(
                n_pos, spec.k, model.d_model, gen, device=h.device, dtype=h.dtype
            )
        else:
            ranked = cache.ranked_ids[layer] if cache is not None else None
            if ranked is None:
                lens_logits = model.unembed(lens.transport(h.float(), layer))
            if spec.selector == "topk":
                tok = (ranked[:, : spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k, excluded=excluded))
            elif spec.selector == "next_k":
                tok = (ranked[:, spec.k : 2 * spec.k] if ranked is not None
                       else select_by_rank(lens_logits, spec.k,
                                           rank_offset=spec.k, excluded=excluded))
            elif spec.selector == "random_lens":
                tok = random_lens_tokens(
                    n_pos, spec.k, clean_logits.shape[-1], gen,
                    excluded=excluded, device=clean_logits.device,
                )
            else:
                raise ValueError(f"unknown selector {spec.selector!r}")
            vecs = lens_vectors(unembed_weight, lens.jacobians[layer].to(h.device), tok)
        bases[layer] = orthonormalise(vecs)

    pos_mask = None
    if spec.positions is not None:
        pos_mask = torch.zeros(n_pos, dtype=torch.bool, device=ids.device)
        pos_mask[list(spec.positions)] = True

    # --- Pass 2: ablated ---
    with _Ablator(model.layers, bases, pos_mask, spec.mode):
        with ActivationRecorder(model.layers, [final]) as rec2:
            model.forward(ids)
            ablated_final = rec2.activations[final][0].detach()

    return AblationResult(
        logits=model.unembed(ablated_final),
        clean_logits=clean_logits,
        effective_rank={l: b.rank for l, b in bases.items()},
        spec=spec,
        ids=ids,
    )


def greedy_match(result: AblationResult, answer_id: int, position: int = -1) -> dict:
    """Score one prompt: did the greedy next token match, clean and ablated?

    The Control A metric per DECISION_control_A §4.4 — greedy next-token
    accuracy against probe-swap.json's ``answer`` field.
    """
    return {
        "clean_correct": int(result.clean_logits[position].argmax()) == answer_id,
        "ablated_correct": int(result.logits[position].argmax()) == answer_id,
    }

## Cell 4 — Write `ablation/tasks.py`

In [ ]:
%%writefile ablation/tasks.py
"""Coarse-output intact tasks — completing `DECISION_control_A.md` §4.5.

§4.5 specified the intact side as pretraining-corpus top-1 match **plus** public
tasks the paper found essentially unaffected at heavy ablation: MMLU, SQuAD,
sentiment, CoLA — "any subset is adequate; two is enough". Only the corpus half
was built for the first Control A run, and it failed (top-1 0.577 at heavy).

That failure is real, but it was measured on the most *sensitive* of the
specified measures. Next-token match over a 151,936-token vocabulary registers
any distributional shift. A multiple-choice task does not: the answer only
changes if the shift specifically reorders four candidate tokens. That asymmetry
is almost certainly why the paper reported those tasks as intact, and it is why
the two measures answer different questions.

**Constrained-choice scoring.** Rather than taking the argmax over the whole
vocabulary, this compares logits only at the first token of each candidate
answer. That is the operation that makes a task coarse.

**These tasks need their own headroom check.** MMLU is 25% at chance, SST-2 is
50%. A model near chance cannot be degraded, and "intact" would then be vacuous
rather than reassuring — the same trap as `diagnose()`'s NO EFFECT branch.
:func:`check_headroom` refuses to certify a task whose clean score is not
clearly above chance.

SQuAD is deliberately excluded: it is extractive and generative, so it cannot be
scored from a single forward pass.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Sequence

import torch

from .harness import AblationSpec, build_cache, run_ablation


@dataclass
class ChoiceTask:
    """A multiple-choice task scored from one forward pass."""

    name: str
    prompts: list[str]
    choices: list[list[str]]      # candidate answer strings, per prompt
    answers: list[int]            # index of the correct choice
    chance: float                 # accuracy of random guessing
    choice_ids: list[list[int]] = field(default_factory=list)

    def tokenise(self, tok: Any) -> None:
        """First token of each candidate, with a leading space.

        Raises if two candidates share a first token — the task would then be
        unscoreable from one forward pass, and silently collapsing them would
        produce a meaningless accuracy rather than an error.
        """
        self.choice_ids = []
        for i, ch in enumerate(self.choices):
            ids = []
            for c in ch:
                t = " " + c.strip()
                try:
                    enc = tok(t, add_special_tokens=False).input_ids
                except TypeError:
                    enc = tok(t).input_ids
                    if torch.is_tensor(enc):
                        enc = enc[0].tolist()
                ids.append(enc[0])
            if len(set(ids)) != len(ids):
                raise ValueError(
                    f"{self.name} item {i}: candidates {ch} do not have distinct "
                    "first tokens, so the task cannot be scored from one forward pass"
                )
            self.choice_ids.append(ids)


def mmlu_prompt(item: dict) -> str:
    letters = "ABCD"
    body = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(item["choices"]))
    return (f"The following are multiple choice questions (with answers) about "
            f"{item.get('subject','general knowledge').replace('_',' ')}.\n\n"
            f"{item['question']}\n{body}\nAnswer:")


def load_mmlu(n: int = 150, seed: int = 0) -> ChoiceTask:
    from datasets import load_dataset
    ds = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=seed).select(range(n))
    return ChoiceTask(
        name="mmlu",
        prompts=[mmlu_prompt(x) for x in ds],
        choices=[["A", "B", "C", "D"] for _ in ds],
        answers=[int(x["answer"]) for x in ds],
        chance=0.25,
    )


def load_sst2(n: int = 150, seed: int = 0) -> ChoiceTask:
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/sst2", split="validation").shuffle(seed=seed).select(range(n))
    return ChoiceTask(
        name="sst2",
        prompts=[f"Review: {x['sentence'].strip()}\nSentiment:" for x in ds],
        choices=[["negative", "positive"] for _ in ds],
        answers=[int(x["label"]) for x in ds],
        chance=0.50,
    )


@torch.no_grad()
def score_task(
    model: Any, lens: Any, unembed_weight: torch.Tensor, task: ChoiceTask,
    specs: dict[str, AblationSpec], *, all_layers: Sequence[int], k_max: int = 20,
    max_seq_len: int = 512, verbose: bool = True, checkpoint_every: int = 25,
    on_checkpoint=None,
) -> dict[str, dict[str, Any]]:
    """Score every condition on the task, looping prompts outer.

    Prompt-outer because MMLU prompts are long: caching all of them would hold
    ``[n_positions, vocab]`` clean logits per prompt and run to tens of GB. One
    cache at a time, reused across all conditions for that prompt, is both
    bounded in memory and free of redundant clean passes.
    """
    if not task.choice_ids:
        task.tokenise(model.tokenizer)

    hits = {name: 0 for name in specs}
    flips = {name: 0 for name in specs}          # answer changed vs the clean pass
    n_done = 0

    for i, (prompt, cids, ans) in enumerate(zip(task.prompts, task.choice_ids, task.answers)):
        cache = build_cache(model, lens, prompt, all_layers, k_max=k_max,
                            max_seq_len=max_seq_len)
        clean_choice = None
        for name, spec in specs.items():
            r = run_ablation(model, lens, unembed_weight, prompt, spec,
                             cache=cache, max_seq_len=max_seq_len)
            # constrained choice: compare ONLY the candidate tokens
            sel = int(torch.tensor([r.logits[-1, c] for c in cids]).argmax())
            hits[name] += (sel == ans)
            if spec.selector == "none":
                clean_choice = sel
            elif clean_choice is not None:
                flips[name] += (sel != clean_choice)
        n_done += 1
        if verbose and n_done % checkpoint_every == 0:
            print(f"  [{n_done}/{len(task.prompts)}] " +
                  "  ".join(f"{n}={hits[n]/n_done:.3f}" for n in list(specs)[:3]))
            if on_checkpoint:
                on_checkpoint({n: hits[n] / n_done for n in specs}, n_done)

    return {name: {"acc": hits[name] / n_done, "k": hits[name], "n": n_done,
                   "answer_flip_rate": flips[name] / n_done if name != "clean" else 0.0}
            for name in specs}


def check_headroom(clean_acc: float, chance: float, *, margin: float = 0.15) -> dict[str, Any]:
    """Can this task serve as an intact-side control at all?

    A task at chance cannot be degraded, so "the ablated model still scores at
    chance" is not evidence of preservation. Mirrors the headroom gate applied to
    the degrading eval, and the NO EFFECT branch of ``intact.diagnose``.
    """
    ok = clean_acc >= chance + margin
    return {
        "clean_acc": clean_acc, "chance": chance, "margin_required": margin,
        "usable": ok,
        "reading": (
            f"Clean {clean_acc:.1%} vs chance {chance:.1%}. Usable as an intact "
            "control: there is room to fall."
            if ok else
            f"Clean {clean_acc:.1%} is not clearly above chance {chance:.1%}. "
            "This task CANNOT serve as an intact-side control — a preserved "
            "score would be indistinguishable from guessing. Report the clean "
            "number and exclude the task from the intact verdict."
        ),
    }


def summarise(results: dict[str, dict], task: ChoiceTask,
              *, primary: str = "heavy|topk") -> dict[str, Any]:
    """Retention relative to clean, plus the headroom gate."""
    clean = results["clean"]["acc"]
    head = check_headroom(clean, task.chance)
    out: dict[str, Any] = {"task": task.name, "clean": clean, "chance": task.chance,
                           "headroom": head, "conditions": {}}
    for name, r in results.items():
        if name == "clean":
            continue
        # retention above chance: what fraction of the model's ABOVE-CHANCE
        # performance survived. Raw accuracy overstates preservation, because a
        # fully destroyed model still scores at chance rather than zero.
        above = clean - task.chance
        out["conditions"][name] = {
            "acc": r["acc"], "retention_raw": r["acc"] / clean if clean else None,
            "retention_above_chance": ((r["acc"] - task.chance) / above) if above > 0 else None,
            "answer_flip_rate": r["answer_flip_rate"],
        }
    if primary in out["conditions"]:
        out["primary"] = {primary: out["conditions"][primary]}
    return out

## Cell 5 — Write `run_intact_tasks.py`

In [ ]:
%%writefile run_intact_tasks.py
"""Coarse-output intact tasks for Control A — completing DECISION_control_A §4.5.

The first Control A run measured the intact side only on WikiText next-token
match, which failed (top-1 0.577 at heavy). That is the most sensitive of the
measures §4.5 specified. This runs the coarse ones the paper actually reported
as unaffected.

Reported as SUPPLEMENTARY. The pre-registered criterion (prereg §4: WikiText
top-1 >= 0.90) failed and that stands; nothing here changes it. These tasks were
named in the signed spec from the start, so running them is completing an
under-delivered spec, not shopping for a measure that passes.

Usage:
    python run_intact_tasks.py --model Qwen/Qwen3-8B \
        --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
        --out results/raw/controlA_qwen3-8b/intact_tasks/
"""
from __future__ import annotations
import argparse, json, time
from dataclasses import asdict
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import jlens
from jlens.lens import JacobianLens
from ablation.harness import AblationSpec, prepare_lens
from ablation.tasks import check_headroom, load_mmlu, load_sst2, score_task, summarise

LENS_REPO = "neuronpedia/jacobian-lens"

# Same bands as the signed prereg + Amendment 003.
STRENGTHS = {
    "light":       tuple(range(20, 24)),
    "medium":      tuple(range(20, 28)),
    "heavy":       tuple(range(20, 32)),
    "heavy-early": tuple(range(15, 32)),
    "heavy-late":  tuple(range(24, 32)),
    "heavy-paper": tuple(range(13, 32)),
}


def build_specs(n_random: int, base_seed: int) -> dict[str, AblationSpec]:
    """Candidates at every strength, plus matched randoms at the primary band.

    Randoms only at `heavy`: they exist here to show what a same-size,
    same-layer ablation does to a coarse task, and repeating that at all six
    strengths would multiply runtime without adding evidence.
    """
    specs = {"clean": AblationSpec(layers=(), k=0, selector="none")}
    for name, layers in STRENGTHS.items():
        specs[f"{name}|topk"] = AblationSpec(layers=layers, k=10, selector="topk")
    specs["heavy|next_k"] = AblationSpec(layers=STRENGTHS["heavy"], k=10, selector="next_k")
    for d in range(n_random):
        seed = base_seed + 1000 * d
        specs[f"heavy|random_lens|{d}"] = AblationSpec(
            layers=STRENGTHS["heavy"], k=10, selector="random_lens", seed=seed)
        specs[f"heavy|random_iso|{d}"] = AblationSpec(
            layers=STRENGTHS["heavy"], k=10, selector="random_iso", seed=seed)
    return specs


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--lens-file", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--n-items", type=int, default=150)
    ap.add_argument("--n-random", type=int, default=5)
    ap.add_argument("--base-seed", type=int, default=20260729)
    ap.add_argument("--max-seq-len", type=int, default=512)
    ap.add_argument("--tasks", default="mmlu,sst2")
    args = ap.parse_args()

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    hf = AutoModelForCausalLM.from_pretrained(args.model, dtype=getattr(torch, args.dtype),
                                              device_map=device)
    lm = jlens.from_hf(hf, AutoTokenizer.from_pretrained(args.model))
    lens = prepare_lens(JacobianLens.from_pretrained(LENS_REPO, filename=args.lens_file), device)
    wu = lm._lm_head.weight.detach()

    specs = build_specs(args.n_random, args.base_seed)
    all_layers = tuple(sorted({l for ls in STRENGTHS.values() for l in ls}))
    print(f"{len(specs)} conditions; layers {all_layers[0]}..{all_layers[-1]}\n")

    loaders = {"mmlu": load_mmlu, "sst2": load_sst2}
    everything = {}

    for tname in args.tasks.split(","):
        tname = tname.strip()
        print(f"=== {tname} ===")
        task = loaders[tname](args.n_items, seed=args.base_seed)
        task.tokenise(lm.tokenizer)
        print(f"  {len(task.prompts)} items, chance {task.chance:.0%}")

        t0 = time.perf_counter()
        res = score_task(lm, lens, wu, task, specs, all_layers=all_layers,
                         max_seq_len=args.max_seq_len)
        summ = summarise(res, task)
        print(f"  {time.perf_counter()-t0:.0f}s\n")

        h = summ["headroom"]
        print(f"  HEADROOM: {h['reading']}")
        if not h["usable"]:
            print("  >>> This task is EXCLUDED from the intact verdict.\n")
        else:
            print()
            print(f"  {'condition':<26}{'acc':>8}{'ret(raw)':>10}{'ret(>chance)':>14}{'flips':>8}")
            print(f"  {'clean':<26}{summ['clean']:>8.3f}{'-':>10}{'-':>14}{'-':>8}")
            for n, c in summ["conditions"].items():
                if n.startswith("heavy|random") and not n.endswith("|0"):
                    continue                      # print one of each for brevity
                print(f"  {n:<26}{c['acc']:>8.3f}{c['retention_raw']:>10.3f}"
                      f"{c['retention_above_chance']:>14.3f}{c['answer_flip_rate']:>8.3f}")
            rl = [summ["conditions"][f"heavy|random_lens|{i}"]["retention_above_chance"]
                  for i in range(args.n_random)]
            ri = [summ["conditions"][f"heavy|random_iso|{i}"]["retention_above_chance"]
                  for i in range(args.n_random)]
            print(f"\n  mean retention(>chance): random_lens {sum(rl)/len(rl):.3f}  "
                  f"random_iso {sum(ri)/len(ri):.3f}  "
                  f"candidate {summ['conditions']['heavy|topk']['retention_above_chance']:.3f}")
        everything[tname] = {"summary": summ, "raw": res}
        (out / f"{tname}.json").write_text(json.dumps(everything[tname], indent=2))

    (out / "_intact_tasks.json").write_text(json.dumps(
        {"results": everything, "config": vars(args),
         "note": "SUPPLEMENTARY. The pre-registered intact criterion "
                 "(WikiText top-1 >= 0.90) failed at 0.577 and that stands."},
        indent=2))
    print(f"\nwrote {out}/_intact_tasks.json")
    print("\nRead alongside the WikiText result, not instead of it. Corpus next-token")
    print("prediction and constrained multiple choice answer different questions;")
    print("a gap between them is a finding about the ablation's shape, not a tie-break.")


if __name__ == "__main__":
    main()

## Cell 6 — Run

Downloads MMLU and SST-2 the first time. Prints the headroom check per task before any results — if a task is at chance, it says so and excludes itself.

In [ ]:
!python run_intact_tasks.py \
    --model Qwen/Qwen3-8B \
    --lens-file qwen3-8b/jlens/Salesforce-wikitext/Qwen3-8B_jacobian_lens.pt \
    --out results/raw/controlA_qwen3-8b/intact_tasks/ \
    --dtype bfloat16 \
    --n-items 150 \
    --n-random 5 \
    --tasks mmlu,sst2

## Cell 7 — The comparison that matters

Puts the coarse tasks next to the WikiText result already on disk. **Retention above chance** is the honest measure: a destroyed 4-way model scores 25%, not 0%, so raw retention flatters it.

In [ ]:
import json

T = json.load(open("results/raw/controlA_qwen3-8b/intact_tasks/_intact_tasks.json"))
try:
    W = json.load(open("results/raw/controlA_qwen3-8b/heavy__topk.json"))["intact"]
    wiki = W["top1_match"]
except Exception:
    wiki = None

print("INTACT SIDE AT HEAVY (L20-31)\n")
if wiki is not None:
    print(f"  WikiText top-1 match      {wiki:.3f}   (pre-registered criterion, "
          f"threshold 0.90 -> {'PASS' if wiki>=0.90 else 'FAIL'})")
for name, blk in T["results"].items():
    s = blk["summary"]
    if not s["headroom"]["usable"]:
        print(f"  {name:<25} EXCLUDED - {s['headroom']['reading']}")
        continue
    c_ = s["conditions"]["heavy|topk"]
    rl = [s["conditions"][f"heavy|random_lens|{i}"]["retention_above_chance"] for i in range(5)]
    print(f"  {name:<25} clean {s['clean']:.3f}  candidate {c_['acc']:.3f}  "
          f"retention(>chance) {c_['retention_above_chance']:.3f}  "
          f"random {sum(rl)/len(rl):.3f}")

print("\nIf the coarse tasks retain most of their above-chance performance while")
print("WikiText next-token match collapses, the ablation reshapes the output")
print("distribution without destroying task competence. That is a specific,")
print("reportable finding - and it is NOT the pre-registered criterion passing.")

## Cell 8 — Download

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("intact_tasks", "zip",
                    "results/raw/controlA_qwen3-8b/intact_tasks")
files.download("intact_tasks.zip")

---
## Report back

Paste the cell 6 output — headroom checks first, then the retention tables — and cell 7.

### How to read whatever comes back

**Coarse tasks largely retained, WikiText collapsed** → the ablation reshapes the output distribution without destroying task competence. Selectivity holds on coarse measures and fails on the sensitive one. Report both; the pre-registered criterion still failed.

**Coarse tasks also collapse** → general damage, unambiguously. The workspace interpretation is not supported at this band and k, and Control A is a clean partial with a well-understood boundary.

**A task lands at chance when clean** → it is excluded automatically. It tells you nothing either way, and the exclusion is the correct behaviour rather than a problem.